# 06 — Frozen ChemBERTa molecular embeddings for hERG

**Purpose:** one additional research comparison against our frozen RF–Hybrid baseline, without changing the submitted model. Two experiments: (A) ChemBERTa embedding → logistic regression; (B) ChemBERTa embedding + Morgan 2048 + 10 descriptors → Random Forest.

**Leakage control:** original 6,841 training molecules and **exact same 3 `StratifiedGroupKFold` scaffold-aware folds** used in Notebook 04. The 1,074 development and 2,085 locked-test molecules are **not evaluated**. The encoder is frozen (no fine-tuning); classifier and any scaler are fitted on each fold's training subset only.

**Important:** this is a *research-only pilot*. Frozen embeddings need an encoder at inference for novel SMILES. A standalone RF ONNX that accepts precomputed ChemBERTa embeddings is **not** a valid replacement for the professor's submission unless the whole featurization/deployment pipeline passes all restrictions (50 MB ONNX, 150M torch parameters, CPU training/runtime, 2D fixed-size input).

## 0. Dependencies and environment

Use the existing `cdd203-hw1` Jupyter kernel. This notebook requires `torch==2.2.2`, `transformers==4.46.3`, `rdkit==2023.9.6`, NumPy, pandas and scikit-learn. **Do not run `pip install -U`**; preserve professor-approved versions. UCSF Jupyter previously had no outbound access to Hugging Face/PyPI. If download fails, download the model repository on your Mac (or an approved computer) and upload a local folder to `CDD203-HW1`, then change `MODEL_SOURCE` below to that directory. Do not publish the coursework dataset.

Primary checkpoint: [DeepChem/ChemBERTa-77M-MTR](https://huggingface.co/DeepChem/ChemBERTa-77M-MTR); fallback: [DeepChem/ChemBERTa-77M-MLM](https://huggingface.co/DeepChem/ChemBERTa-77M-MLM). We load the **backbone** via `AutoModel`, not the regression head, and inspect reported missing/unexpected weights. If the backbone doesn't load appropriately, stop and investigate; do not silently use an uninitialized model.

In [ ]:
from pathlib import Path
import os, time, json, warnings
import numpy as np
import pandas as pd
from IPython.display import display
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone

RDLogger.DisableLog('rdApp.warning')
SEED = 42
DATA = Path('hw1data.tab')
OUT = Path('research_outputs/chemberta_pilot')
OUT.mkdir(parents=True, exist_ok=True)
assert DATA.exists(), 'Open this notebook with CDD203-HW1 as working directory.'

import torch
import transformers
from transformers import AutoTokenizer, AutoModel
print('torch:',torch.__version__, 'transformers:', transformers.__version__)
print('device: CPU (intentional)')
assert torch.__version__.startswith('2.2.2'), 'Use professor-approved torch 2.2.2'
assert transformers.__version__ == '4.46.3', 'Use professor-approved transformers 4.46.3'
torch.set_num_threads(min(2, os.cpu_count() or 2))


## 1. Reproduce original split exactly and compute RF Hybrid features

Same `GroupShuffleSplit` calls and Morgan/descriptor order as Notebook 04. **No resplitting**.

In [ ]:
df = pd.read_csv(DATA, sep='\t')
assert {'Drug','Y'}.issubset(df.columns)
smiles = df['Drug'].astype(str).tolist()
mols = [Chem.MolFromSmiles(s) for s in smiles]
assert all(m is not None for m in mols)
y = df['Y'].to_numpy(dtype=int)
groups = np.array([Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(m),canonical=True) for m in mols])
idx = np.arange(len(df))
trainval_idx, test_idx = next(GroupShuffleSplit(n_splits=1,test_size=.20,random_state=SEED).split(idx,y,groups))
a,b = next(GroupShuffleSplit(n_splits=1,test_size=.125,random_state=SEED).split(trainval_idx,y[trainval_idx],groups[trainval_idx]))
train_idx=trainval_idx[a]; val_idx=trainval_idx[b]
assert (len(train_idx),len(val_idx),len(test_idx))==(6841,1074,2085)
assert set(groups[train_idx]).isdisjoint(groups[val_idx])
assert set(groups[train_idx]).isdisjoint(groups[test_idx])

smiles_train=[smiles[i] for i in train_idx]
y_train=y[train_idx]
g_train=groups[train_idx]
GEN=rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=2048)
def hybrid_featurize(ss):
    out=np.zeros((len(ss),2058),dtype=np.float32)
    for i,s in enumerate(ss):
        m=Chem.MolFromSmiles(s)
        if m is None: raise ValueError('Invalid SMILES: '+str(s))
        DataStructs.ConvertToNumpyArray(GEN.GetFingerprint(m),out[i,:2048])
        out[i,2048:]=[
            Descriptors.MolWt(m),Descriptors.MolLogP(m),rdMolDescriptors.CalcTPSA(m),
            rdMolDescriptors.CalcNumHBD(m),rdMolDescriptors.CalcNumHBA(m),
            rdMolDescriptors.CalcNumRotatableBonds(m),rdMolDescriptors.CalcNumAromaticRings(m),
            rdMolDescriptors.CalcFractionCSP3(m),Chem.GetFormalCharge(m),
            sum(atom.GetAtomicNum()==7 for atom in m.GetAtoms())]
    return out

X_hybrid=hybrid_featurize(smiles_train)
folds=list(StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=SEED).split(X_hybrid,y_train,g_train))
assert all(not(set(g_train[tr]) & set(g_train[va])) for tr,va in folds)
print('TRAIN:',X_hybrid.shape,'DEV (unused):',len(val_idx),'LOCKED (unused):',len(test_idx))
print('Fold sizes:',[(len(tr),len(va)) for tr,va in folds])


## 2. Load pretrained encoder and verify weights

On UCSF, if Internet is restricted, set `MODEL_SOURCE` to a locally uploaded folder containing `config.json`, tokenizer files and model weights. We use CPU and keep the entire transformer frozen.

In [ ]:
MODEL_ID = 'DeepChem/ChemBERTa-77M-MTR'
MODEL_SOURCE = MODEL_ID  # or 'chemberta-77m-mtr' for a local model directory
# Fallback if MTR backbone loading fails: 'DeepChem/ChemBERTa-77M-MLM'
BATCH_SIZE = 32
MAX_LENGTH = 256
LOCAL_FILES_ONLY = Path(MODEL_SOURCE).exists()

start = time.perf_counter()
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_SOURCE, local_files_only=LOCAL_FILES_ONLY, trust_remote_code=False)
    encoder, info = AutoModel.from_pretrained(
        MODEL_SOURCE, local_files_only=LOCAL_FILES_ONLY,
        trust_remote_code=False, output_loading_info=True
    )
except Exception as e:
    raise RuntimeError('Unable to load ChemBERTa. On UCSF, download the full HF repository on a machine with internet and set MODEL_SOURCE to its uploaded folder. Original error: '+repr(e)) from e

missing = info.get('missing_keys', [])
unexpected = info.get('unexpected_keys', [])
print('Missing keys (first 15):',missing[:15], 'count:',len(missing))
print('Unexpected keys (first 15):',unexpected[:15], 'count:',len(unexpected))
# A regression/classification head may be unexpected when extracting the encoder, which is fine.
# Missing weights INSIDE the encoder backbone, however, invalidate this experiment.
backbone_missing=[k for k in missing if not any(x in k for x in ['pooler','classifier','lm_head','regression'])]
assert not backbone_missing, 'Backbone weights were missing: inspect model checkpoint / architecture before continuing.'
encoder.eval().to('cpu')
for p in encoder.parameters(): p.requires_grad_(False)
num_params=sum(p.numel() for p in encoder.parameters())
print('Encoder parameters:',num_params, '(<150M:',num_params<150_000_000,')')
assert num_params < 150_000_000
print('Load seconds:',round(time.perf_counter()-start,2))


## 3. Extract frozen SMILES embeddings once

Masked-mean pooling over non-padding tokens. This is a design choice, not guaranteed to outperform CLS pooling. No labels or validation outcomes enter representation extraction. We cache the embeddings so experiments can rerun without transformer inference. We explicitly verify SMILES order and model identity before reusing a cache.

In [ ]:
import hashlib
smiles_hash=hashlib.sha256('\n'.join(smiles_train).encode()).hexdigest()
model_tag=hashlib.sha256((str(MODEL_SOURCE)+'|'+str(MAX_LENGTH)+'|masked_mean').encode()).hexdigest()[:12]
cache_file=OUT/f'embeddings_{model_tag}.npz'

@torch.no_grad()
def embed_smiles(ss):
    chunks=[]
    for start in range(0,len(ss),BATCH_SIZE):
        batch=ss[start:start+BATCH_SIZE]
        toks=tokenizer(batch,return_tensors='pt',padding=True,truncation=True,max_length=MAX_LENGTH)
        if start==0:
            print('First batch token lengths:',toks['attention_mask'].sum(axis=1).tolist()[:8])
        result=encoder(**toks,return_dict=True)
        h=result.last_hidden_state
        mask=toks['attention_mask'].unsqueeze(-1).to(h.dtype)
        pooled=(h*mask).sum(dim=1)/mask.sum(dim=1).clamp(min=1)
        chunks.append(pooled.cpu().numpy().astype(np.float32))
        if ((start//BATCH_SIZE)+1) % 50==0:
            print('Embedded:',min(start+BATCH_SIZE,len(ss)),'/',len(ss))
    return np.vstack(chunks)

if cache_file.exists():
    z=np.load(cache_file,allow_pickle=False)
    assert str(z['smiles_hash'].item())==smiles_hash, 'Cached dataset order differs. Delete stale cache or change tag.'
    embeddings=z['embeddings']
    print('Loaded cached embeddings:',cache_file)
else:
    t0=time.perf_counter()
    embeddings=embed_smiles(smiles_train)
    np.savez_compressed(cache_file,embeddings=embeddings,smiles_hash=np.array(smiles_hash))
    print('Embedding seconds:',round(time.perf_counter()-t0,1))
assert embeddings.shape[0]==len(y_train) and np.isfinite(embeddings).all()
print('Embedding shape:',embeddings.shape)


## 4. Three fair scaffold folds: A versus B versus frozen RF baseline

All classifiers train within each outer fold. The pretrained encoder is **not fine-tuned**, so computing its fixed embeddings before folds is okay. Scalers in A are fitted within the fold. Hyperparameters are pre-specified; no search on outer validation results.

In [ ]:
RF_PARAMS=dict(n_estimators=150,min_samples_leaf=1,max_features=0.2,
               max_depth=30,class_weight='balanced',random_state=SEED,n_jobs=2)
models={
    'RF-Hybrid (matched baseline)': (X_hybrid,RandomForestClassifier(**RF_PARAMS)),
    'ChemBERTa embedding + LR': (embeddings,make_pipeline(StandardScaler(),LogisticRegression(C=1.0,max_iter=2000,random_state=SEED))),
    'ChemBERTa + Morgan + descriptors + RF': (np.hstack([X_hybrid,embeddings]).astype(np.float32),RandomForestClassifier(**RF_PARAMS)),
}
results=[]
for name,(X,base) in models.items():
    for fid,(tr,va) in enumerate(folds,1):
        est=clone(base)
        t0=time.perf_counter(); est.fit(X[tr],y_train[tr]); fit_s=time.perf_counter()-t0
        t0=time.perf_counter(); prob=est.predict_proba(X[va])[:,1]; pred_s=time.perf_counter()-t0
        results.append(dict(model=name,fold=fid,n=len(va),AUROC=roc_auc_score(y_train[va],prob),
                            AUPRC=average_precision_score(y_train[va],prob),
                            fit_seconds=fit_s,predict_seconds=pred_s))
        print(name,'fold',fid,'AUROC',round(results[-1]['AUROC'],4),'AUPRC',round(results[-1]['AUPRC'],4))
result_df=pd.DataFrame(results)
summary=result_df.groupby('model',as_index=False).agg(mean_AUROC=('AUROC','mean'),sd_AUROC=('AUROC','std'),
    mean_AUPRC=('AUPRC','mean'),mean_fit_seconds=('fit_seconds','mean'))
display(summary.round(4).sort_values('mean_AUROC',ascending=False))
result_df.to_csv(OUT/'frozen_chemberta_fold_results.csv',index=False)
summary.to_csv(OUT/'frozen_chemberta_summary.csv',index=False)


## 5. CPU featurization benchmark (required for realistic deployment decision)

An encoder is not free at inference time. This tests **SMILES tokenization + frozen transformer embedding** for a fixed set of training SMILES only. It is not an ONNX benchmark, and model sizes are not yet validated. The RF baseline requires only RDKit features.

In [ ]:
sample_smiles=smiles_train[:min(128,len(smiles_train))]
t0=time.perf_counter(); hybrid_small=hybrid_featurize(sample_smiles); rf_features_seconds=time.perf_counter()-t0
t0=time.perf_counter(); embedding_small=embed_smiles(sample_smiles); encoder_seconds=time.perf_counter()-t0
print(pd.DataFrame([
  {'method':'Morgan+descriptors','molecules':len(sample_smiles),'seconds':rf_features_seconds,'molecules_per_second':len(sample_smiles)/max(rf_features_seconds,1e-9)},
  {'method':'ChemBERTa tokenization+encoder','molecules':len(sample_smiles),'seconds':encoder_seconds,'molecules_per_second':len(sample_smiles)/max(encoder_seconds,1e-9)}
]).round(3).to_string(index=False))


## 6. Interpretation and stopping rule

- Compare scaffold CV AUROC and AUPRC to the **matched RF baseline above**, not the earlier Locked Test AUROC (`0.8806`).
- The experiment is exploratory, since previous research used these folds extensively; a small improvement is not strong independent evidence.
- If the gains are modest, **keep submitted RF-Hybrid**. If major improvements are consistent across folds, evaluate independent external data and end-to-end ONNX packaging, file size, and CPU speed *before* considering a replacement.
- **Do not reuse the 2,085 locked molecules for selecting a model, and do not assert this research pipeline satisfies the assignment submission contract.**
- If model loading fails in UCSF due to network restrictions, follow the local model instructions above or run this pilot in an approved environment. The dataset must not be uploaded to a third-party service without permission.

**Model citations:** [MTR](https://huggingface.co/DeepChem/ChemBERTa-77M-MTR) and [MLM](https://huggingface.co/DeepChem/ChemBERTa-77M-MLM).